# Latent Signal Encoder Comparison

Main comparison of linear, residual nonlinear, and regularized encoders on the original synthetic paired-view datasets.
This notebook covers the noise-only, linear-signal, and cubic-signal latent experiments, plus the branch-history and similarity-matrix diagnostics.


## 1. Imports

The path setup lets this notebook work when Jupyter starts either in the repository root or inside `experiment-notebooks/`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

GLOBAL_SEED = 123
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

P_DIM = 128
Q_DIM = 128

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / "contrastive_encoders").exists():
        module_root = candidate
        break
else:
    raise RuntimeError("Could not find the contrastive_encoders package folder.")

if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

PLOT_DIR = module_root / "reports" / "report-plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("module_root:", module_root)
print("plot output:", PLOT_DIR)


In [ ]:
import importlib

import contrastive_encoders
from contrastive_encoders import (
    architectures,
    data,
    experiments,
    losses,
    interpretability,
    metrics,
    plotting,
    regularization,
    training,
)

for module in [
    architectures,
    data,
    losses,
    interpretability,
    metrics,
    regularization,
    training,
    experiments,
    plotting,
]:
    importlib.reload(module)

importlib.reload(contrastive_encoders)

from contrastive_encoders import (
    TrainConfig,
    cubic_coordinate_probe_curve,
    friendly_results_table,
    generate_deterministic_relation_dataset,
    make_experiment_datasets,
    make_first_experiment_configs,
    make_model_spec_table,
    paper_contrastive_loss,
    plot_alpha_sweep_curve,
    plot_branch_ratio_history,
    plot_coordinate_probe_curves,
    plot_deterministic_snr_sweep,
    plot_metric_by_config,
    plot_signal_noise_sweep,
    plot_signal_recovery_by_config,
    plot_similarity_heatmap,
    plot_top5_retrieval_by_setting,
    plot_train_test_separation_by_setting,
    run_alpha_sweep,
    run_deterministic_relation_experiment,
    run_first_experiment,
    run_signal_noise_sweep,
    train_one_model_with_artifacts,
)


## 2. Mathematical objective

For embeddings `z_x = f_X(X)` and `z_y = f_Y(Y)`, define

```text
s_ij = <z_x_i, z_y_j>
```

The paper objective maximizes

```text
L = sum_i s_ii
    - sum_i log sum_j exp(s_ij)
    - sum_i log sum_j exp(s_ji)
```

The training code minimizes `-L / N` using `paper_contrastive_loss(...)`.

Important: this is **not** the standard cross-entropy InfoNCE replacement. By default it uses raw inner products, no loss-side normalization, and temperature `1.0`. The encoder architecture now has internal branch normalization so `alpha` controls the nonlinear correction, but the paper loss itself is unchanged.


## 3. Model comparison grid

The pure linear baseline is now a normalized linear encoder:

```text
g(u) = Normalize(G u)
```

The residual nonlinear models normalize both branches so `alpha` controls the nonlinear correction:

```text
g(u) = Normalize(G u) + alpha * Normalize(A1 sigma(A2 u + b))
```

The normalization layers use `BatchNorm1d(..., affine=False)`. The `affine=False` part is important because it prevents a learned scale parameter from secretly undoing `alpha`.

This also keeps the pure linear embedding scale controlled before the raw-dot-product loss sees it. So `alpha=0` means no nonlinear correction, while larger `alpha` lets the normalized MLP correction contribute more strongly.


In [ ]:
configs = make_first_experiment_configs(epochs=300)
model_table = make_model_spec_table(configs, p=P_DIM, q=Q_DIM)
display(model_table)


## 4. Run pure-noise, linear-signal, and cubic-signal experiments

The experiment creates three datasets:

- `Noise only`: X and Y are independent noise, so there is no real paired signal.
- `Linear signal`: X has latent `Z_x`, Y has latent `Z_y`, and the relationship is `Z_y = Z_x`.
- `Cubic signal`: X has latent `Z_x`, Y has latent `Z_y`, and the relationship is `Z_y = standardized(Z_x ** 3)`.

This is closer to the caption/image analogy: the two views are different random datasets, but their hidden variables are related when a signal is present. Every model uses the same paper contrastive loss, so differences should come from model capacity and regularization.


In [ ]:
results = run_first_experiment(
    configs=configs,
    seed=GLOBAL_SEED,
    device=DEVICE,
    n_train=160,
    n_test=800,
    p=P_DIM,
    q=Q_DIM,
    signal_strength=2.0,
    noise_std=1.0,
)

results


## 5. Focused comparison table

These are the main metrics in friendlier language:

- **True-pair separation**: true X/Y pairs score higher than mismatched pairs. Higher is better.
- **Shuffled-pair check**: the same score after breaking the X/Y pairing. This should stay near zero.
- **Exact-pair accuracy**: how often the closest Y is the true paired Y. Higher is better.
- **Top-5 retrieval accuracy**: how often the true paired Y is among the 5 closest Y candidates. Higher is better.
- **X embedding vs Z_x**: correlation between the X embedding and the true X latent.
- **X embedding vs Z_y**: correlation between the X embedding and the paired Y latent. In the cubic setting, this means `Z_y = standardized(Z_x ** 3)`.
- **Y embedding vs Z_y**: correlation between the Y embedding and the true Y latent.
- **Ridge-probe R^2**: percent of held-out latent variation explained by a simple ridge regression from the learned embedding.
- **Mean nonlinear/linear ratio**: diagnostic check that the measured nonlinear branch size follows `alpha`.


In [ ]:
summary_columns = [
    "setting",
    "config",
    "architecture",
    "parameter_count",
    "parameter_count_per_train_sample",
    "train_pair_separation",
    "test_pair_separation",
    "shuffled_pair_separation",
    "test_pair_match_accuracy",
    "test_top5_pair_match_accuracy",
    "test_best_view_correlation",
    "x_signal_recovery",
    "x_related_signal_recovery",
    "y_signal_recovery",
    "x_probe_r2_z_x",
    "x_probe_r2_z_y",
    "y_probe_r2_z_y",
    "mean_nonlinear_to_linear_ratio",
]

summary_table = friendly_results_table(results[summary_columns].round(4))
display(summary_table)


## 6. Main plots

These plots separate the questions:

- Does the model memorize training pairs without generalizing?
- Does true-pair separation survive on held-out pairs?
- Is the true paired Y inside the top 5 retrieved candidates?
- Does the embedding recover `Z_x`, the transformed paired latent `Z_y`, or both?
- How much held-out latent variation is explained by a ridge probe from the embedding?
- Does the measured branch contribution follow `alpha`?

Each plot is also saved as a PNG in `reports/report-plots/` with a readable file name.


In [ ]:
for setting_name in ["Noise only", "Linear signal", "Cubic signal"]:
    plot_train_test_separation_by_setting(
        results,
        setting=setting_name,
        save_dir=PLOT_DIR,
    )

plot_metric_by_config(
    results,
    setting="Noise only",
    metrics=[
        "test_pair_separation",
        "shuffled_pair_separation",
    ],
    title="Noise only: shuffled-pair sanity check",
    ylabel="True-pair separation",
    save_dir=PLOT_DIR,
)
plot_top5_retrieval_by_setting(
    results,
    setting="Noise only",
    save_dir=PLOT_DIR,
)

for setting_name in ["Linear signal", "Cubic signal"]:
    plot_metric_by_config(
        results,
        setting=setting_name,
        metrics=[
            "test_pair_separation",
            "shuffled_pair_separation",
        ],
        title=f"{setting_name}: held-out true-pair separation",
        ylabel="True-pair separation",
        save_dir=PLOT_DIR,
    )

    plot_top5_retrieval_by_setting(
        results,
        setting=setting_name,
        save_dir=PLOT_DIR,
    )

    plot_signal_recovery_by_config(
        results,
        setting=setting_name,
        title=f"{setting_name}: X/Y embeddings versus true latents",
        save_dir=PLOT_DIR,
    )

plot_metric_by_config(
    results,
    setting="Cubic signal",
    metrics=[
        "mean_nonlinear_to_linear_ratio",
    ],
    title="Final alpha diagnostic: measured nonlinear-to-linear branch ratio",
    ylabel="Nonlinear branch norm / linear branch norm",
    save_dir=PLOT_DIR,
)


## 7. Branch history and similarity heatmaps

This section retrains the residual models on both the linear-signal and cubic-signal datasets so we can inspect training dynamics, not just final metrics.

The branch-history plot tracks:

```text
|| nonlinear branch || / || linear branch ||
```

over epochs. The heatmap plots `S = z_x z_y^T` for a small held-out batch. A clear bright diagonal means the model gives true X/Y pairs higher similarity than mismatched pairs. These figures are saved to `reports/report-plots/`.


In [ ]:
diagnostic_datasets = make_experiment_datasets(
    seed=GLOBAL_SEED,
    n_train=160,
    n_test=800,
    p=P_DIM,
    q=Q_DIM,
    signal_strength=2.0,
    noise_std=1.0,
)

diagnostic_configs = [
    config for config in configs
    if config.architecture == "residual"
]

diagnostic_settings = ["Linear signal", "Cubic signal"]
diagnostic_runs = {}
all_history_frames = []

for setting_index, setting_name in enumerate(diagnostic_settings):
    diagnostic_runs[setting_name] = {}
    setting_history_frames = []

    for config_index, config in enumerate(diagnostic_configs):
        run = train_one_model_with_artifacts(
            dataset=diagnostic_datasets[setting_name],
            config=config,
            seed=GLOBAL_SEED + 200 + 100 * setting_index + config_index,
            device=DEVICE,
            standardize=True,
            history_interval=25,
        )
        diagnostic_runs[setting_name][config.name] = run

        history = pd.DataFrame(run.history)
        history["setting"] = setting_name
        history["config"] = config.name
        setting_history_frames.append(history)
        all_history_frames.append(history)

    setting_branch_history = pd.concat(setting_history_frames, ignore_index=True)
    plot_branch_ratio_history(
        setting_branch_history,
        title=f"{setting_name}: nonlinear/linear branch ratio over training",
        save_dir=PLOT_DIR,
    )

selected_heatmap_model = "MLP nonlinear (alpha=0.10)"

for setting_name in diagnostic_settings:
    heatmap_run = diagnostic_runs[setting_name][selected_heatmap_model]
    plot_similarity_heatmap(
        heatmap_run.z_x_test,
        heatmap_run.z_y_test,
        n=40,
        title=f"{setting_name}: held-out similarity matrix for {selected_heatmap_model}",
        save_dir=PLOT_DIR,
    )

branch_history = pd.concat(all_history_frames, ignore_index=True)
